In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 1. 기본 설정

from pyspark.sql import functions as F
from pyspark.sql.types import (
    DecimalType,
    ArrayType,
    StructType,
    StructField,
    StringType,
    DoubleType,
)

# ==============================
# Delta Lake Source 설정
# ==============================


SOURCE_PREDICTION_TABLE = (
    f"dt4_team1_databricks.gold.farm_risk_predictions_20251216"
)

print("Source Delta Table:", SOURCE_PREDICTION_TABLE)

# 선택 사항: 정규화한 결과를 Delta Lake에도 저장
DELTA_RISK_SCORE_TABLE = "dt4_team1_databricks.gold.farm_risk_scores_export"
DELTA_XAI_FACTOR_TABLE = "dt4_team1_databricks.gold.farm_xai_factors_export"

# ==============================
# PostgreSQL Target 설정
# ==============================

POSTGRES_RISK_SCORE_TABLE = "public.farm_risk_scores"
POSTGRES_XAI_FACTOR_TABLE = "public.farm_xai_factors"

pg_host = "dt4-postgresql.postgres.database.azure.com"
pg_port = "5432"
pg_database = "bioroute_db"

jdbc_url = f"jdbc:postgresql://{pg_host}:{pg_port}/{pg_database}?sslmode=require"

# ⚠️ 최종 Job에서는 하드코딩하지 말고 secrets 사용 추천
# pg_user = dbutils.secrets.get(scope="postgres", key="user")
# pg_password = dbutils.secrets.get(scope="postgres", key="password")

pg_user = "azureuser"
pg_password = "rootborn#1"

connection_properties = {
    "user": pg_user,
    "password": pg_password,
    "driver": "org.postgresql.Driver",
}

print("Source Delta Table:", SOURCE_PREDICTION_TABLE)
print("Target Risk Score Table:", POSTGRES_RISK_SCORE_TABLE)
print("Target XAI Factor Table:", POSTGRES_XAI_FACTOR_TABLE)

In [ ]:
# MAGIC %md
# MAGIC # 2. PostgreSQL 연결 테스트

connection_test_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option("query", "SELECT 1 AS connection_test")
    .option("user", pg_user)
    .option("password", pg_password)
    .option("driver", "org.postgresql.Driver")
    .load()
)

display(connection_test_df)

In [ ]:
# MAGIC %md
# MAGIC # 3. Delta Lake 예측 결과 읽기

prediction_df = spark.table(SOURCE_PREDICTION_TABLE)

display(prediction_df.limit(20))

print("Prediction rows:", prediction_df.count())
print("Columns:", prediction_df.columns)
prediction_df.printSchema()

In [ ]:
# MAGIC %md
# MAGIC # 4. 필수 컬럼 검증

required_cols = [
    "farmId",
    "riskDate",
    "riskScore",
    "riskLevel",
    "modelVersion",
    "lastUpdated",
]

missing_cols = [c for c in required_cols if c not in prediction_df.columns]

if missing_cols:
    raise ValueError(f"Delta prediction table에 필수 컬럼이 없습니다: {missing_cols}")

has_risk_factors = "riskFactors" in prediction_df.columns

print("riskFactors column exists:", has_risk_factors)

In [ ]:
# MAGIC %md
# MAGIC # 5. farm_risk_scores 저장용 DataFrame 생성

from pyspark.sql.window import Window

risk_score_df = (
    prediction_df
    .select(
        F.col("farmId").cast("string").alias("farm_id"),
        F.to_date(F.col("riskDate")).alias("risk_date"),
        F.round(F.col("riskScore").cast("double"), 2)
            .cast(DecimalType(5, 2))
            .alias("risk_score"),
        F.col("riskLevel").cast("string").alias("risk_level"),
        F.col("modelVersion").cast("string").alias("model_version"),
        F.col("lastUpdated").cast("timestamp").alias("calculated_at"),
    )
    .filter(F.col("farm_id").isNotNull())
    .filter(F.col("risk_date").isNotNull())
    .filter(F.col("risk_score").isNotNull())
)

risk_score_df = (
    risk_score_df
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("farm_id", "risk_date", "model_version")
            .orderBy(F.col("calculated_at").desc_nulls_last())
        )
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

display(risk_score_df.limit(20))

print("Risk score rows:", risk_score_df.count())

In [ ]:
# MAGIC %md
# MAGIC # 6. 저장 전 검증

risk_score_check_df = (
    risk_score_df
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("farm_id").alias("farm_count"),
        F.min("risk_score").alias("min_risk_score"),
        F.max("risk_score").alias("max_risk_score"),
        F.min("risk_date").alias("min_risk_date"),
        F.max("risk_date").alias("max_risk_date"),
        F.min("calculated_at").alias("min_calculated_at"),
        F.max("calculated_at").alias("max_calculated_at"),
    )
)

display(risk_score_check_df)

In [ ]:
# MAGIC %md
# MAGIC # 7. 정규화 결과를 Delta Lake에도 저장

(
    risk_score_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DELTA_RISK_SCORE_TABLE)
)

print("Saved Delta table:", DELTA_RISK_SCORE_TABLE)

In [ ]:
# MAGIC %md
# MAGIC # 8. PostgreSQL 기존 데이터 읽기
# MAGIC 
# MAGIC 같은 farm_id + risk_date + model_version 데이터가 이미 있으면 중복 insert를 막기 위한 단계

existing_risk_score_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option(
        "query",
        f"""
        SELECT
            risk_score_id,
            farm_id,
            risk_date,
            risk_score,
            risk_level,
            model_version,
            calculated_at
        FROM {POSTGRES_RISK_SCORE_TABLE}
        """
    )
    .option("user", pg_user)
    .option("password", pg_password)
    .option("driver", "org.postgresql.Driver")
    .load()
)

display(existing_risk_score_df.limit(20))
print("Existing risk score rows:", existing_risk_score_df.count())

In [ ]:
# MAGIC %md
# MAGIC # 9. 신규 farm_risk_scores만 PostgreSQL에 저장

new_risk_score_df = (
    risk_score_df.alias("n")
    .join(
        existing_risk_score_df
        .select("farm_id", "risk_date", "model_version")
        .alias("e"),
        on=["farm_id", "risk_date", "model_version"],
        how="left_anti"
    )
)

display(new_risk_score_df.limit(20))

print("New risk score rows to insert:", new_risk_score_df.count())

In [ ]:
if new_risk_score_df.count() > 0:
    (
        new_risk_score_df.write
        .format("jdbc")
        .option("url", jdbc_url)
        .option("dbtable", POSTGRES_RISK_SCORE_TABLE)
        .option("user", pg_user)
        .option("password", pg_password)
        .option("driver", "org.postgresql.Driver")
        .mode("append")
        .save()
    )

    print(f"Saved {new_risk_score_df.count()} rows to {POSTGRES_RISK_SCORE_TABLE}")
else:
    print("No new risk score rows to insert.")

In [ ]:
# MAGIC %md
# MAGIC # 10. PostgreSQL에서 risk_score_id 다시 읽기
# MAGIC 
# MAGIC farm_xai_factors에 넣으려면 risk_score_id가 필요하므로, 저장된 parent row를 다시 읽는다.

saved_risk_score_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option(
        "query",
        f"""
        SELECT
            risk_score_id,
            farm_id,
            risk_date,
            risk_score,
            risk_level,
            model_version,
            calculated_at
        FROM {POSTGRES_RISK_SCORE_TABLE}
        """
    )
    .option("user", pg_user)
    .option("password", pg_password)
    .option("driver", "org.postgresql.Driver")
    .load()
)

# 이번 Delta 결과와 매칭되는 risk_score_id만 가져오기
risk_score_with_id_df = (
    risk_score_df.alias("r")
    .join(
        saved_risk_score_df.alias("s"),
        on=["farm_id", "risk_date", "model_version"],
        how="inner"
    )
    .select(
        F.col("s.risk_score_id").alias("risk_score_id"),
        F.col("r.farm_id").alias("farm_id"),
        F.col("r.risk_date").alias("risk_date"),
        F.col("r.risk_score").alias("risk_score"),
        F.col("r.risk_level").alias("risk_level"),
        F.col("r.model_version").alias("model_version"),
        F.col("r.calculated_at").alias("calculated_at"),
    )
)

display(risk_score_with_id_df.limit(20))
print("Risk score rows with id:", risk_score_with_id_df.count())

In [ ]:
# MAGIC %md
# MAGIC # 11. riskFactors 컬럼 구조 확인

if has_risk_factors:
    prediction_df.select("riskFactors").printSchema()
    display(prediction_df.select("farmId", "riskDate", "modelVersion", "riskFactors").limit(10))
else:
    print("riskFactors 컬럼이 없으므로 farm_xai_factors 저장 단계는 건너뜁니다.")

In [ ]:
# MAGIC %md
# MAGIC # 12. farm_xai_factors 저장용 DataFrame 생성
# MAGIC 
# MAGIC riskFactors가 array<struct> 형태이든 JSON string 형태이든 최대한 처리되도록 구성

risk_factor_schema = ArrayType(
    StructType([
        StructField("factorCode", StringType(), True),
        StructField("factor_code", StringType(), True),
        StructField("label", StringType(), True),
        StructField("icon", StringType(), True),
        StructField("weight", DoubleType(), True),
    ])
)

if has_risk_factors:
    risk_factors_raw_df = prediction_df.select(
        F.col("farmId").cast("string").alias("farm_id"),
        F.to_date(F.col("riskDate")).alias("risk_date"),
        F.col("modelVersion").cast("string").alias("model_version"),
        F.col("lastUpdated").cast("timestamp").alias("created_at"),
        F.col("riskFactors").alias("riskFactors"),
    )

    risk_factors_dtype = dict(prediction_df.dtypes).get("riskFactors")

    print("riskFactors dtype:", risk_factors_dtype)

    # riskFactors가 string이면 JSON으로 파싱
    if risk_factors_dtype == "string":
        risk_factors_parsed_df = risk_factors_raw_df.withColumn(
            "riskFactorsParsed",
            F.from_json(F.col("riskFactors"), risk_factor_schema)
        )
    else:
        risk_factors_parsed_df = risk_factors_raw_df.withColumn(
            "riskFactorsParsed",
            F.col("riskFactors")
        )

    exploded_factor_df = (
        risk_factors_parsed_df
        .withColumn("factor", F.explode_outer(F.col("riskFactorsParsed")))
        .select(
            "farm_id",
            "risk_date",
            "model_version",
            "created_at",
            F.coalesce(
                F.col("factor.factorCode"),
                F.col("factor.factor_code")
            ).cast("string").alias("factor_code"),
            F.col("factor.label").cast("string").alias("label"),
            F.col("factor.icon").cast("string").alias("icon"),
            F.col("factor.weight").cast("double").alias("weight"),
        )
        .filter(F.col("factor_code").isNotNull())
    )

    # risk_score_id 붙이기
    xai_factor_df = (
        exploded_factor_df.alias("x")
        .join(
            risk_score_with_id_df
            .select("risk_score_id", "farm_id", "risk_date", "model_version")
            .alias("r"),
            on=["farm_id", "risk_date", "model_version"],
            how="inner"
        )
        .select(
            F.col("factor_code"),
            F.col("farm_id"),
            F.col("risk_score_id").cast("long").alias("risk_score_id"),
            F.col("label"),
            F.col("icon"),
            F.round(F.col("weight"), 6).cast(DecimalType(10, 6)).alias("weight"),
            F.coalesce(F.col("created_at"), F.current_timestamp()).alias("created_at"),
        )
    )

    display(xai_factor_df.limit(20))
    print("XAI factor rows:", xai_factor_df.count())

else:
    xai_factor_df = None

In [ ]:
# MAGIC %md
# MAGIC # 13. farm_xai_factors 검증

if xai_factor_df is not None:
    xai_check_df = (
        xai_factor_df
        .agg(
            F.count("*").alias("row_count"),
            F.countDistinct("farm_id").alias("farm_count"),
            F.countDistinct("risk_score_id").alias("risk_score_count"),
            F.countDistinct("factor_code").alias("factor_code_count"),
            F.min("weight").alias("min_weight"),
            F.max("weight").alias("max_weight"),
        )
    )

    display(xai_check_df)
else:
    print("xai_factor_df가 없습니다.")

In [ ]:
# MAGIC %md
# MAGIC # 14. XAI 결과를 Delta Lake에도 저장

if xai_factor_df is not None:
    (
        xai_factor_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(DELTA_XAI_FACTOR_TABLE)
    )

    print("Saved Delta table:", DELTA_XAI_FACTOR_TABLE)
else:
    print("No XAI factor Delta table saved.")

In [ ]:
# MAGIC %md
# MAGIC # 15. 기존 farm_xai_factors 읽기
# MAGIC 
# MAGIC 중복 insert 방지를 위해 risk_score_id + factor_code 기준으로 기존 데이터 확인

if xai_factor_df is not None:
    existing_xai_factor_df = (
        spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .option(
            "query",
            f"""
            SELECT
                xai_factor_id,
                factor_code,
                farm_id,
                risk_score_id,
                label,
                icon,
                weight,
                created_at
            FROM {POSTGRES_XAI_FACTOR_TABLE}
            """
        )
        .option("user", pg_user)
        .option("password", pg_password)
        .option("driver", "org.postgresql.Driver")
        .load()
    )

    display(existing_xai_factor_df.limit(20))
    print("Existing XAI factor rows:", existing_xai_factor_df.count())
else:
    existing_xai_factor_df = None

In [ ]:
# MAGIC %md
# MAGIC # 16. 신규 farm_xai_factors만 PostgreSQL에 저장

if xai_factor_df is not None:
    new_xai_factor_df = (
        xai_factor_df.alias("n")
        .join(
            existing_xai_factor_df
            .select("risk_score_id", "factor_code")
            .alias("e"),
            on=["risk_score_id", "factor_code"],
            how="left_anti"
        )
    )

    display(new_xai_factor_df.limit(20))
    print("New XAI factor rows to insert:", new_xai_factor_df.count())

    if new_xai_factor_df.count() > 0:
        (
            new_xai_factor_df.write
            .format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", POSTGRES_XAI_FACTOR_TABLE)
            .option("user", pg_user)
            .option("password", pg_password)
            .option("driver", "org.postgresql.Driver")
            .mode("append")
            .save()
        )

        print(f"Saved {new_xai_factor_df.count()} rows to {POSTGRES_XAI_FACTOR_TABLE}")
    else:
        print("No new XAI factor rows to insert.")

else:
    print("riskFactors가 없어서 farm_xai_factors insert를 건너뜁니다.")

In [ ]:
# MAGIC %md
# MAGIC # 17. PostgreSQL 저장 결과 확인 - farm_risk_scores

verify_risk_score_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option(
        "query",
        f"""
        SELECT
            risk_score_id,
            farm_id,
            risk_date,
            risk_score,
            risk_level,
            model_version,
            calculated_at
        FROM {POSTGRES_RISK_SCORE_TABLE}
        ORDER BY calculated_at DESC
        LIMIT 50
        """
    )
    .option("user", pg_user)
    .option("password", pg_password)
    .option("driver", "org.postgresql.Driver")
    .load()
)

display(verify_risk_score_df)

In [ ]:
# MAGIC %md
# MAGIC # 18. PostgreSQL 저장 결과 확인 - farm_xai_factors

verify_xai_factor_df = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .option(
        "query",
        f"""
        SELECT
            x.xai_factor_id,
            x.factor_code,
            x.farm_id,
            x.risk_score_id,
            r.risk_date,
            r.risk_score,
            r.risk_level,
            x.label,
            x.icon,
            x.weight,
            x.created_at
        FROM {POSTGRES_XAI_FACTOR_TABLE} x
        LEFT JOIN {POSTGRES_RISK_SCORE_TABLE} r
            ON x.risk_score_id = r.risk_score_id
        ORDER BY x.created_at DESC
        LIMIT 100
        """
    )
    .option("user", pg_user)
    .option("password", pg_password)
    .option("driver", "org.postgresql.Driver")
    .load()
)

display(verify_xai_factor_df)